# DFD video training — actor-disjoint and rollback-safe

This notebook is the preferred walkthrough for the high-accuracy video candidate. It never overwrites the active model. Actors 01–19 are training, 20–23 validation, and 24–28 locked test. Manipulated pairs crossing actor pools are discarded to prevent identity leakage.

Locked test result: **89.2% accuracy**, **89.9% balanced accuracy**, **97.8% AUC** on 130 videos.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "dfd" / "raw"
FRAMES = ROOT / "data" / "dfd" / "frames"
ARTIFACTS = ROOT / "artifacts" / "dfd_candidate"
print("Repository:", ROOT)
print("Raw DFD directory:", RAW)

## Dataset layout

Extract the original videos directly into `data/dfd/raw/` and the manipulated archive so it creates `data/dfd/raw/DFD_manipulated_sequences/`. The archives and extracted media are intentionally excluded from Git.

In [ ]:
POOLS = {
    "train": set(range(1, 20)),
    "val": set(range(20, 24)),
    "test": set(range(24, 29)),
}

def split_for(actor_ids):
    for split, pool in POOLS.items():
        if all(actor in pool for actor in actor_ids):
            return split
    return None

assert split_for([1, 2]) == "train"
assert split_for([24, 28]) == "test"
assert split_for([19, 20]) is None  # cross-pool pair is rejected

## Extract four YOLO face crops per included video

In [ ]:
# Runs the complete reproducible preprocessing module.
%run ../training/prepare_dfd.py

## Training recipe

EfficientNet-B4 starts from a FaceForensics++ checkpoint. Epoch 1 calibrates the head; epochs 2–4 fine-tune end-to-end. Sampling is class-balanced. Blur, JPEG recompression, crops, color variation and horizontal flips are applied only to training.

In [ ]:
import torch
from torch import nn
from torchvision import models

class DFDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b4(weights=None)
        features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(features, 2))

    def forward(self, x):
        return self.head(self.backbone(x))

DFDModel()

In [ ]:
# Trains, calibrates threshold on validation, then evaluates the locked test once.
%run ../training/train_dfd_b4.py

## Inspect the saved deployment-gate report

In [ ]:
report_path = ROOT / "reports" / "dfd_training_report.json"
report = json.loads(report_path.read_text())
report["passed_deployment_gate"], report["test"]